# 08 — Final Results & Verdict

This notebook reads the headline numbers produced by `run_all.py` (`results/headline_results.json`) and states the conclusion plainly.

If you have not run the full pipeline yet, run `python run_all.py` from the project root first.

In [1]:
# --- standard setup (run me first) ---
import sys, os
# Make the project root importable so `from src import ...` works from notebooks/.
sys.path.insert(0, os.path.abspath(".."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("Setup complete. Project root:", os.path.abspath(".."))

Setup complete. Project root: C:\Users\Taffy Jackson\leveraged-trend-following


In [2]:
import json
from src import config
with open(config.RESULTS_DIR / 'headline_results.json') as f:
    H = json.load(f)
print('Sample:', H['data']['underlying_start'], '->', H['data']['underlying_end'],
      f"({H['data']['years']:.1f} years of {H['data']['underlying_ticker']})")

Sample: 1988-01-04 -> 2026-06-08 (38.4 years of ^SP500TR)


## Baselines vs the best leverage at the 200-day window

In [3]:
import pandas as pd
bh = H['part1_buy_and_hold']; ma = H['part2_ma_to_cash_200d']
lev = H['part3_leveraged_200d']
tbl = pd.DataFrame({
  'Buy & Hold':   {k: bh[k] for k in ['cagr','volatility','sharpe','max_drawdown','calmar']},
  'MA200 -> Cash':{k: ma[k] for k in ['cagr','volatility','sharpe','max_drawdown','calmar']},
  'Lev 1.5x':     lev['1.5x'], 'Lev 2x': lev['2x'], 'Lev 3x': lev['3x'],
}).T
tbl

,cagr,volatility,sharpe,max_drawdown,calmar
Buy & Hold,0.1147,0.1787,0.5356,-0.5525,0.2075
MA200 -> Cash,0.1017,0.1176,0.6434,-0.2055,0.4951
Lev 1.5x,0.1157,0.2336,0.4653,-0.7070,0.1637
Lev 2x,0.1121,0.2937,0.4134,-0.8303,0.1350
Lev 3x,0.0896,0.4204,0.3492,-0.9612,0.0932


## Did any leveraged config beat buy-and-hold on everything?

In [4]:
print('Configs beating buy & hold on ALL of CAGR/Sharpe/Calmar/drawdown:')
print('  gross (0 cost):', H['part4_beats_all_count_gross'])
print('  net (realistic):', H['part4_beats_all_count_net'])
print()
print('Fraction of historical periods where 2x leverage beat buy & hold (total return):',
      f"{H['part5_periods_leverage_helped_frac']:.0%}")

Configs beating buy & hold on ALL of CAGR/Sharpe/Calmar/drawdown:
  gross (0 cost): 0
  net (realistic): 0

Fraction of historical periods where 2x leverage beat buy & hold (total return): 33%


## The Monte Carlo verdict at an S&P-like point

In [5]:
sp = H['part7_sp_like_point']
print(f"S&P-like world: drift={sp['drift']:.0%}, vol={sp['vol']:.0%}")
print('Median CAGR by leverage :', sp['median_cagr_by_leverage'])
print('P(beat 1x) by leverage  :', sp['prob_beat_1x_by_leverage'])
print('Optimal leverage (iid)  :', H['part7_sp_like_optimal_leverage'])

S&P-like world: drift=11%, vol=18%
Median CAGR by leverage : {'1x': 0.0993, '1.25x': 0.1199, '1.5x': 0.1386, '2x': 0.1696, '2.5x': 0.1919, '3x': 0.2044}
P(beat 1x) by leverage  : {'1x': 0.0, '1.25x': 0.9039, '1.5x': 0.8891, '2x': 0.8592, '2.5x': 0.8257, '3x': 0.7848}
Optimal leverage (iid)  : 3.0


## The verdict

Putting it together (your exact numbers may differ slightly by data vintage):

1. **Does leverage in bad markets beat buy-and-hold?** Not on a risk-adjusted basis. Zero configurations beat buy-and-hold on all four metrics at once. Low leverage (1.25–1.5x) can nudge CAGR up a little, but always with a *disproportionately* deeper drawdown.

2. **Does it beat the moving-average-to-cash rule?** No. The classic Faber rule has the best Sharpe and by far the shallowest drawdowns. Adding leverage moves you in the wrong direction.

3. **Which moving average / leverage is 'best'?** ~200 days is the most robust window; the best *risk-adjusted* leverage is **1x** (i.e. no leverage). Leverage only ever helped in specific fast-rebound regimes (post-2009 dips, the COVID crash), not on average.

4. **When does volatility decay destroy returns?** Whenever volatility is high — exactly the below-trend regimes the strategy leverages. The Monte Carlo map shows leverage only pays in low-vol, positive-drift markets.

**Bottom line:** the idea is intuitive but the data rejects it. It is a good example of a hypothesis that sounds clever and fails an honest test. The full discussion is in `reports/research_paper.md`.